# **Spotify Dataset — Data Engineer**

## **Import Libraries**

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

## **Project path Configuration**

In [65]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"

## **Load the Dataset**

In [66]:
df = pd.read_csv(RAW_DATA_PATH)

### Basic Dataset Check

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [68]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## **Data Cleaning**

### Remove Unnecessary Index Column

This section removes automatically generated index columns that are not useful for analysis.

In [69]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head(1)

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,acoustic


### **Check Missing Values**
This section handles missing values in the dataset.

- Columns with excessive missing values are removed to maintain data quality.  
- After that, rows containing remaining missing values are dropped to ensure consistency for further analysis.

In [70]:
missing_values = df.isnull().sum()

missing_table = missing_values[missing_values > 0].to_frame(name='Missing Count')

print("Columns with missing values:")
print(missing_table)

Columns with missing values:
            Missing Count
artists                 1
album_name              1
track_name              1


### Remove columns with excessive missing values

Columns with a missing value ratio greater than 40% are removed to improve dataset reliability and reduce noise in further analysis.

In [71]:
col_threshold = 0.4    
df = df.loc[:, df.isnull().mean() < col_threshold]
df.shape

(114000, 20)

### Remove remaining Missing values

After removing low-quality columns, rows containing remaining missing values are dropped to ensure dataset consistency.

In [72]:
df = df.dropna()

### Verify missing values after cleaning

This step verifies that the dataset no longer contains missing values after the cleaning process.

In [73]:
df.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

### **Duplicate**
This section checks duplicated records in the dataset.

In [74]:
duplicated_rows = df.duplicated().sum()

print(f"Number of duplicated rows: {duplicated_rows}")

Number of duplicated rows: 450


In [75]:
df.drop_duplicates(inplace=True) 

In [76]:
df.reset_index(drop=True, inplace=True)

### **Deduplicate by track_id**
This section groups repeated tracks so each track_id appears once in the cleaned dataset.

In [77]:
if "track_id" in df.columns:
    original_columns = df.columns.tolist()
    duplicated_track_ids = df.duplicated(subset=["track_id"]).sum()
    print(f"Number of duplicated track_id rows: {duplicated_track_ids}")

    def most_common_value(series):
        mode_values = series.mode(dropna=True)
        if not mode_values.empty:
            return mode_values.iloc[0]
        non_missing_values = series.dropna()
        if not non_missing_values.empty:
            return non_missing_values.iloc[0]
        return np.nan

    aggregation_rules = {}

    mean_columns = [
        "popularity",
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo"]
    for col in mean_columns:
        if col in df.columns:
            aggregation_rules[col] = "mean"

    if "duration_ms" in df.columns:
        aggregation_rules["duration_ms"] = "median"

    if "explicit" in df.columns:
        aggregation_rules["explicit"] = "max"

    mode_columns = ["key", "mode", "time_signature", "track_genre"]
    for col in mode_columns:
        if col in df.columns:
            aggregation_rules[col] = most_common_value

    first_columns = ["artists", "album_name", "track_name"]
    for col in first_columns:
        if col in df.columns:
            aggregation_rules[col] = "first"

    for col in df.columns:
        if col != "track_id" and col not in aggregation_rules:
            aggregation_rules[col] = "first"

    df = df.groupby("track_id", as_index=False).agg(aggregation_rules)
    df = df[[col for col in original_columns if col in df.columns]]
    df.reset_index(drop=True, inplace=True)

    print(f"Rows after track_id deduplication: {df.shape[0]}")
    print(f"Remaining duplicated track_id rows: {df.duplicated(subset=['track_id']).sum()}")
else:
    print("Column 'track_id' was not found. Skipping track_id deduplication.")

Number of duplicated track_id rows: 23809
Rows after track_id deduplication: 89740
Remaining duplicated track_id rows: 0


### **Outlier Detection**

This section identifies numerical columns for outlier processing using the IQR method.
- Remove rows having MANY outlier features only

In [78]:
continuous_cols = [
    "duration_ms",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"]
continuous_cols = [col for col in continuous_cols if col in df.columns]


**Outlier Removal Strategy** 
Instead of removing observations containing a single outlier value, this study only removes rows that exhibit multiple outlier features simultaneously. This approach helps preserve potentially meaningful musical variations while reducing the influence of extreme abnormal observations.

### Remove Outliers Using IQR

In [79]:
outlier_flags = pd.DataFrame(index=df.index)

for col in continuous_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_flags[col] = (df[col] < lower_bound) | (df[col] > upper_bound)

# Remove only rows that are outliers in multiple continuous features
outlier_count = outlier_flags.sum(axis=1)

df = df[outlier_count <= 2].copy()
df.reset_index(drop=True, inplace=True)

df.shape

(88237, 20)

## **Final Dataset Check**

This section verifies the final dataset structure after preprocessing and cleaning.

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88237 entries, 0 to 88236
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          88237 non-null  str    
 1   artists           88237 non-null  str    
 2   album_name        88237 non-null  str    
 3   track_name        88237 non-null  str    
 4   popularity        88237 non-null  float64
 5   duration_ms       88237 non-null  float64
 6   explicit          88237 non-null  bool   
 7   danceability      88237 non-null  float64
 8   energy            88237 non-null  float64
 9   key               88237 non-null  int64  
 10  loudness          88237 non-null  float64
 11  mode              88237 non-null  int64  
 12  speechiness       88237 non-null  float64
 13  acousticness      88237 non-null  float64
 14  instrumentalness  88237 non-null  float64
 15  liveness          88237 non-null  float64
 16  valence           88237 non-null  float64
 17  temp

In [81]:
df.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [82]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0000vdREvCVMxbQTkS888c,Rill,Lolly,Lolly,44.0,160725.0,True,0.910,0.374,8,-9.844,0,0.1990,0.075700,0.00301,0.1540,0.432,104.042,4,german
1,000CC8EParg64OmTxVnZ0p,Glee Cast,Glee Love Songs,It's All Coming Back To Me Now (Glee Cast Vers...,47.0,322933.0,False,0.269,0.516,0,-7.361,1,0.0366,0.406000,0.00000,0.1170,0.341,178.174,4,club
2,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,X,Böxig Leise - Pig & Dan Remix,22.0,515360.0,False,0.686,0.560,5,-13.264,0,0.0462,0.001140,0.18100,0.1110,0.108,119.997,4,minimal-techno
3,000RDCYioLteXcutOjeweY,Jordan Sandhu,Teeje Week,Teeje Week,62.0,190203.0,False,0.679,0.770,0,-3.537,1,0.1900,0.058300,0.00000,0.0825,0.839,161.721,4,hip-hop
4,000qpdoc97IMTBvF8gwcpy,Paul Kalkbrenner,Zeit,Tief,19.0,331240.0,False,0.519,0.431,6,-13.606,0,0.0291,0.000964,0.72000,0.0916,0.234,129.971,4,minimal-techno


## Export Cleaned Dataset

In [83]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
df.to_csv(OUTPUT_PATH,index=False)